<a href="https://colab.research.google.com/github/ys23-lys/ESAA/blob/main/ESAA_OB_WEEK1_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**주제** 부동산 허위 매물 분류 해커톤

https://dacon.io/competitions/official/236439/codeshare/12230?page=1&dtype=recent

**코드 흐름**

**전처리**

결측치 제거: 결측치를 열의 평균값으로 대치.

이상치 제거: DBSCAN의 sklearn.cluster를 이용해 노이즈(-1)만 필터링.

레이블 인코딩

다중공선성 확인: heatmap상 0.8 이상, vif 계수 10 이상을 다중공선성이 있다고 판단함.
- **vif 계수**: 분산 팽창 요인 1/(1-R^2)

**예측**

XGBoost 이용

RandomizedSearchCV로 하이퍼파라미터 추정

**새롭게 알게 된 점/어려운 점/배울 점**

다중공선성 확인 과정에서 다중공선성이 있다고 판단하는 기준을 알게 되었으며, 이때 heatmap상 상관계수로 판단하는 방법뿐 아니라 분산 팽창 요인인 vif 계수를 이용하는 방법도 있다는 것을 새롭게 알게 되었다.

In [ ]:
### 전처리 과정 중 이상치 제거

from sklearn.cluster import DBSCAN

scaler = StandardScaler()
scaled_data = scaler.fit_transform(train[['보증금', '월세', '총주차대수', '관리비']])

# DBSCAN 실행
dbscan = DBSCAN(eps=2.5, min_samples=5)
clusters = dbscan.fit_predict(scaled_data)

# 노이즈(-1)만 필터링
train['cluster'] = clusters
train = train[train['cluster'] != -1].drop(columns=['cluster'])
train

In [ ]:
### 전처리 과정 중 다중공선성 확인

# 상수항 추가
con = add_constant(X)

# VIF 계산
vif = pd.DataFrame()
vif["Variable"] = con.columns
vif["VIF"] = [variance_inflation_factor(con.values, i) for i in range(con.shape[1])]

print(vif)